# nanoVLM-RSI quickstart

A small VLM improves itself on verifiable multimodal tasks: it acts, a verifier scores what it did,
GRPO (written from scratch) turns that into a LoRA update, and a gate keeps the update only if it
helps on a separate gate split. Held-out tasks are never used to decide anything.

Runtime: **GPU** (Runtime → Change runtime type → T4 or better) for the real run; the first smoke
cell also works on CPU.

In [ ]:
REPO = "https://github.com/patrickamadeus/nanovlm-rsi"   # set to the repository you cloned this notebook from
!git clone -q $REPO nanovlm-rsi
%cd nanovlm-rsi
!pip -q install -e ".[plot]"

## 1. Close a loop on CPU with a tiny random policy (~1 min)
This checks the plumbing, not learning: a randomly initialized model can't solve anything.

In [ ]:
!python rsi.py --tiny --iters 2 --run_dir runs/smoke

## 2. GRPO on held-out counting

SmolVLM-256M counts blue squares in procedurally generated images. The self-curriculum keeps it
practising tasks at its competence frontier; every update has to clear the gate.

In [ ]:
!python rsi.py --update_rule weights --proposer self --static_families count_objects \
    --iters 30 --tasks_per_iter 8 --group_size 8 --eval_every 5 --eval_n 200 --gate_n 32 \
    --run_dir runs/colab

## 3. Success against cost
Always plot against cost, not iterations: carriers differ wildly in how much compute an update costs.

In [ ]:
!python tools/plot.py runs/colab --x gpu_seconds --out colab.png
from IPython.display import Image
Image("colab.png")

## 4. What happened, update by update

In [ ]:
from infra.trace import Trace
tr = Trace("runs/colab")
for u in tr.records("update"):
    print(u["iter"], "committed" if u["committed"] else "reverted ", u.get("reason") or "",
          "gate", u.get("gate_before"), "->", u.get("gate_after"))

Next: `docs/architecture.md` explains every organ and how to add your own environment or carrier, and `walkthrough.py` is the whole method in one file.